![Platform+](images/banner.png)

# Lab 8: Time Travel
## Undo a bad pipeline overwrite — recover a staging table

> **Persona track: Data Engineer** — same lab, your angle.

:point_right: **In this lesson, you'll build a staging table, watch a reload job overwrite it with the wrong data, then recover the good state with Time Travel — the safety net every pipeline needs.**

The scenario: You're a data engineer at **Roastery & Co.** Your nightly job reloads `ORDERS_STAGING` using an `INSERT OVERWRITE`. Tonight the upstream extract was misconfigured — it returned only one store's rows — and the overwrite replaced all your good staging data with the data for a single store. Downstream models are already reading from it. This lab shows you how to roll it back, cleanly and in place.

We'll cover:
1. What Time Travel is and why it's a pipeline's safety net
2. Building your staging table (your own copy of `ORDERS`)
3. Reproducing the incident — a bad `INSERT OVERWRITE` (and capturing its query id)
4. Looking back with `BEFORE` to see the pre-overwrite state
5. Recovering with a zero-copy point-in-time `CLONE`, then swapping it in
6. Your angle — using clones for backfills without duplicating storage

:books: **By the end you'll be able to:**
- Explain the Time Travel retention window and what it covers
- Recover the state before a specific job with `BEFORE(statement => ...)`
- Recover an older state with a zero-copy `CLONE ... BEFORE(...)` and swap it in
- Verify a recovery *before* you overwrite the live table
- Use a point-in-time clone to build a backfill without duplicating storage
- Use CoCo to diagnose and recover from a bad load

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set Your Session Context

Every Snowflake session runs under a role and a warehouse. The cell below sets your everyday role, makes sure your personal warehouse exists, and creates the `TT_LAB` schema in your own database — the safe sandbox where all of this lab's create/overwrite/recover work happens.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.TT_LAB;
USE SCHEMA {{my_db}}.TT_LAB;


---
## 1 · What Time Travel Gives You

[Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel) is Snowflake's built-in history for your data. For a set **retention period**, Snowflake keeps the historical versions of every table, schema, and database — so a bad load, a wrong overwrite, or an accidental drop is recoverable without a backup restore. For a pipeline, it's the safety net that turns "we clobbered production" into a one-minute rollback.

Within the retention window you can:

| Do this | With |
|---|---|
| **Query** a table as it was *just before* a specific statement (e.g. your reload job) | `BEFORE(statement => '<query_id>')` |
| **Query** a table as it was at a point in time | `AT(timestamp => ...)` / `AT(offset => -seconds)` |
| **Recover** an older state without duplicating storage | `CREATE TABLE ... CLONE ... AT/BEFORE(...)` |
| **Bring back** a dropped table, schema, or database | `UNDROP` |

**The retention window:**

| Edition | Default | Maximum |
|---|---|---|
| Standard | 1 day | 1 day |
| Enterprise and above | 1 day | 90 days (per object, via `DATA_RETENTION_TIME_IN_DAYS`) |

> :bulb: **Why engineers care.** `INSERT OVERWRITE`, `TRUNCATE`-and-reload, and `MERGE` are everyday pipeline operations — and every one of them can go wrong when the source is bad. Time Travel means the *previous* version of the table is still queryable, so you can roll back to the last good load instead of re-running an entire pipeline from scratch.

---
## 2 · Build Your Staging Table

You never test reload logic against shared source data — you stage it in your own schema first. The cell below copies `ORDERS` from the read-only `PLATPLUS_DB.ROASTERY_SALES` schema into a staging table **you own**, `TT_LAB.ORDERS_STAGING`. This stands in for the last good nightly load.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_STAGING AS
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

SELECT COUNT(*) AS good_load_rows FROM {{my_db}}.TT_LAB.ORDERS_STAGING;


> :mag_right: **Review the output.** Note `good_load_rows` — this is the last-known-good state of staging, the number you'll recover back to.

### Where the retention window is set. 🕳️

Rollback only works because your staging table sits inside a **Time Travel retention window**, controlled by one object parameter: `DATA_RETENTION_TIME_IN_DAYS`. It can be set at four levels — **account → database → schema → table** — and the **lowest (most specific) level wins**. A table with no value of its own inherits from its schema; the schema from its database; the database from the account default.

Two quick looks tell the whole story. First, `SHOW PARAMETERS` on the **account** reports the default value *and*, in the `level` column, *where* it's set — a **blank `level`** means it's inherited, not pinned. Then a single query against `INFORMATION_SCHEMA` reads the **effective retention** on your database, schema, and staging table side by side:

In [ ]:
-- Account default — the value everything inherits unless overridden
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN ACCOUNT;


In [ ]:
-- Effective retention (in days) on your database, schema, and staging table — side by side.
-- SHOW PARAMETERS tells you WHERE a value is pinned; INFORMATION_SCHEMA tells you the effective days per object.
SELECT * FROM (
  SELECT 1 AS lvl, 'DATABASE' AS scope, DATABASE_NAME AS object_name, RETENTION_TIME AS retention_days
  FROM {{my_db}}.INFORMATION_SCHEMA.DATABASES
  WHERE DATABASE_NAME = CURRENT_DATABASE()
  UNION ALL
  SELECT 2, 'SCHEMA', SCHEMA_NAME, RETENTION_TIME
  FROM {{my_db}}.INFORMATION_SCHEMA.SCHEMATA
  WHERE SCHEMA_NAME = 'TT_LAB'
  UNION ALL
  SELECT 3, 'TABLE', TABLE_NAME, RETENTION_TIME
  FROM {{my_db}}.INFORMATION_SCHEMA.TABLES
  WHERE TABLE_SCHEMA = 'TT_LAB' AND TABLE_NAME = 'ORDERS_STAGING'
) ORDER BY lvl;


> :mag_right: **Review both outputs.** The account `SHOW PARAMETERS` reports **`1`** with a **blank `level`** — the account is running on the default, with nothing pinned above it. The `INFORMATION_SCHEMA` query then confirms the same **1-day** window is effective at the database, schema, and table levels — each inherits from the account because nothing is pinned lower down. (Reading the `level` column: when it names a *higher* scope than the object you asked about, that object is **inheriting** from that scope; when it names the object's own scope, the value is **pinned** there.) Pin `DATA_RETENTION_TIME_IN_DAYS` on the schema or table and its `retention_days` here would rise — the lowest (most specific) setting wins.

> :bulb: **Why two tools?** `SHOW PARAMETERS` tells you *where* a value is set (the `level` column — inherited vs. pinned); `INFORMATION_SCHEMA` tells you the *effective* number of days per object and lets you list many objects at once. Reach for whichever answers the question in front of you.

> 📌 **Clock note.** "1 day" is a *rolling* 24-hour window measured from each change — not a calendar-day boundary. A bad load at 3 PM is recoverable until 3 PM tomorrow. Reference a point more than 24 hours back and the query fails — the history has aged out.

---
## 3 · The Incident — A Bad Overwrite

[`INSERT OVERWRITE`](https://docs.snowflake.com/en/sql-reference/sql/insert#overwriting-a-table) is the idempotent reload workhorse: it truncates the target and inserts the new rows in one atomic step. Tonight the upstream extract was misconfigured and returned **only store 3's** orders — so the overwrite replaces the entire current contents of the table with the orders for a single store.

The cell captures the overwrite's query id into `$bad_overwrite` with [`LAST_QUERY_ID()`](https://docs.snowflake.com/en/sql-reference/functions/last_query_id) — your anchor to the state *just before* the job ran.

> :point_right: **Your turn — fill in the blank.** The `INSERT OVERWRITE` below is the scripted mistake — leave it exactly as-is. Your job is the **fourth line**: capture the id of the reload job that just ran, so Section 4 can look back to the state right before it. Which function returns the id of your most recent query?
>
> **Make sure you edit the fourth line before you execute the next cell.**

In [ ]:
-- The reload job: source was misconfigured and returned only one store's rows
INSERT OVERWRITE INTO {{my_db}}.TT_LAB.ORDERS_STAGING
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS WHERE store_id = 3;
SET bad_overwrite = ___;   -- capture the id of the query that just ran

<details>
<summary>📖 Reference solution</summary>

    SET bad_overwrite = LAST_QUERY_ID();

`LAST_QUERY_ID()` returns the id of the statement that ran immediately before it in this session — the overwrite. You anchor `BEFORE(statement => $bad_overwrite)` to it in Section 4.
</details>

> **Got a red error?** That's expected if you left `___` as-is. Don't use the Fix button — this is your puzzle to solve. The `INSERT OVERWRITE` *already ran* (the good load is gone), so re-running the cell won't help — `LAST_QUERY_ID()` would capture a second overwrite instead of the original.
>
> Expand the **Stumbled?** section below — it has a recovery query that finds the original overwrite's query id from your session history.

In [ ]:
SELECT $bad_overwrite;

<details>
<summary>🛟 Stumbled? Were you able to retrieve the Query ID?</summary>

**What happened:** The `INSERT OVERWRITE` in the cell above already ran — the last good load *is* gone from the live table. But `$bad_overwrite` either doesn't exist (you left `___` as-is and got a syntax error) or holds a junk value (you replaced it with something other than `LAST_QUERY_ID()`). Either way, Section 4's `BEFORE(statement => $bad_overwrite)` can't find the right point in history.

**How to fix it:** Run the SQL below in a new cell. It searches your recent Query History for the actual overwrite statement and captures its id into `$bad_overwrite`:

```sql
SET bad_overwrite = (
  SELECT QUERY_ID FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
  WHERE QUERY_TEXT ILIKE 'INSERT OVERWRITE%ORDERS_STAGING%'
    AND QUERY_TYPE = 'INSERT'
    AND ROWS_PRODUCED > 0
  ORDER BY START_TIME DESC LIMIT 1
);
SELECT $bad_overwrite AS captured_query_id;
```
</details>

In [ ]:
SELECT COUNT(*) AS rows_after_overwrite FROM {{my_db}}.TT_LAB.ORDERS_STAGING;

> :mag_right: **Review the output.** `rows_after_overwrite` is a fraction of `good_load_rows` — the good data was overwritten entirely by the data for store 3. **Don't re-run the pipeline yet.** The previous load is still in Time Travel, and the next section proves it.

> 📌 **Lab shortcut vs. real life.** You captured `$bad_overwrite` in the very cell that caused it — tidy for a lab, but not how it happens in a pipeline. In production the damage surfaces when a downstream model breaks or a dashboard goes flat, and you'd trace the offending job in [Query History](https://docs.snowflake.com/en/user-guide/ui-snowsight-activity#query-history) (filter by warehouse, time window, and target table). CoCo can run that search for you:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Search my recent query history for INSERT OVERWRITE statements against {{my_db}}.TT_LAB.ORDERS_STAGING and return the query_id and timestamp of the most recent one.</pre>

That's how you'd find the offending job for real — CoCo reads `QUERY_HISTORY` and hands you the anchor for `BEFORE(statement => ...)`.

> :warning: **Heads-up on session variables.** The `$bad_overwrite` variable lives in *this* session. If you restart the kernel or reconnect, re-run Section 3 so it's set again before continuing — otherwise the `BEFORE` cell below will error on an unset variable.

---
## 4 · Look Back With `BEFORE`

`BEFORE(statement => ...)` gives you a read-only view of the table *immediately before* a specific statement ran. Because you captured the overwrite's query id, you can point straight at the last good state — no guessing at timestamps.

> :point_right: **Your turn — fill in the blank.** You know which statement clobbered the table (`$bad_overwrite`). Complete the Time Travel clause that shows the staging table's state *immediately before* that job ran.

In [ ]:
-- The state right before the bad overwrite — fill in the Time Travel clause
SELECT COUNT(*) AS rows_before_overwrite
FROM {{my_db}}.TT_LAB.ORDERS_STAGING ___(statement => $bad_overwrite);


<details>
<summary>📖 Reference solution</summary>

    SELECT COUNT(*) AS rows_before_overwrite
    FROM {{my_db}}.TT_LAB.ORDERS_STAGING BEFORE(statement => $bad_overwrite);

`BEFORE(statement => …)` returns the data as it was just before the given query id executed — your last good load.
</details>

> :mag_right: **Review the output.** `rows_before_overwrite` matches your original `good_load_rows` — the last good load is intact in history. Nothing was truly lost.

> :bulb: **`AT` is the other anchor.** When you don't have the query id, `AT(timestamp => ...)` or `AT(offset => -3600)` ("one hour ago") reaches the same history by time instead of by statement — see the [`AT | BEFORE`](https://docs.snowflake.com/en/sql-reference/constructs/at-before) reference. For a known bad job, `BEFORE(statement => query_id)` is the most precise choice.

---
## :robot_face: CoCo Assist

When a load goes wrong, CoCo is your recovery partner — it can find the offending job, explain which Time Travel clause to use, and draft the rollback SQL. Here's how you'd use it in the moment.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — the explanation is where the learning happens.
> 2. **CoCo is interactive** — ask follow-ups until it clicks ("What if the overwrite ran inside a stored procedure?").
> 3. **The prompts below are starting points** — your real learning comes from trying your own.

---

**Example 1: Choose the right recovery clause**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">My pipeline ran an INSERT OVERWRITE that replaced good rows with bad data. Explain how to recover the prior state with Time Travel, and when to use BEFORE(statement) vs AT(timestamp) vs AT(offset).</pre>

---

**Example 2: Find the job that clobbered the table**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Search my query history in INFORMATION_SCHEMA for INSERT OVERWRITE or TRUNCATE statements against ORDERS_STAGING in the last 24 hours. Show the query id, timestamp, and rows affected, then write the CLONE ... BEFORE statement to roll it back.</pre>

---

> :bulb: **These are just examples.** CoCo can help with any recovery question: "Is my retention window long enough to recover this?", "How do I make INSERT OVERWRITE safer next time?", "Clone yesterday's state into a backfill table for me." Ask in plain English and follow up until you're confident.

---
## 5 · Recover With a Point-in-Time Clone

The safe rollback is: **clone the good state into a new table, verify it, then swap it in.** You never overwrite the live staging table until you're sure — and downstream models keep reading the current object name once you rename the good copy into place.

A [`CLONE`](https://docs.snowflake.com/en/sql-reference/sql/create-clone) is **zero-copy** — near-instant, no extra storage until the clone and source diverge. Cloning `BEFORE(statement => $bad_overwrite)` gives you the full pre-overwrite state.

> :point_right: **Your turn — fill in the blank.** Recover the pre-overwrite state into a new table you can inspect. Which keyword makes a **zero-copy** copy of another table (optionally at a past point via `BEFORE`/`AT`)?

In [ ]:
-- Zero-copy clone of the pre-overwrite state into a new table you can inspect
CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_STAGING_RECOVERED
___ {{my_db}}.TT_LAB.ORDERS_STAGING BEFORE(statement => $bad_overwrite);   -- zero-copy copy keyword

SELECT COUNT(*) AS recovered_rows FROM {{my_db}}.TT_LAB.ORDERS_STAGING_RECOVERED;


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_STAGING_RECOVERED
    CLONE {{my_db}}.TT_LAB.ORDERS_STAGING BEFORE(statement => $bad_overwrite);

`CLONE` makes a zero-copy copy; adding `BEFORE(statement => $bad_overwrite)` clones the table exactly as it was before the overwrite.
</details>

> :mag_right: **Review the output.** `recovered_rows` matches `good_load_rows`. The live staging table is still in its damaged state — you've lost nothing by inspecting the clone first.

### Another use: clone a past state for a backfill 📸

Recovery isn't the only use of Time Travel. Because a clone is **zero-copy**, you can cheaply materialise *any* past state as its own table — perfect for a **backfill** or a "what did the data look like before last night's run?" investigation, without touching the live table or duplicating storage. Do this **now, before the swap** — while `ORDERS_STAGING` still carries the history that `$bad_overwrite` points to.

Here you'll clone the pre-overwrite state into a dated backfill table — the kind of artifact you'd hand to a downstream job to re-derive a day's metrics.

> :point_right: **Your turn — write this one yourself.** You just cloned the pre-overwrite state to *recover* it. Now use the same move for a **backfill**: write a zero-copy `CLONE ... BEFORE(statement => $bad_overwrite)` that materialises that state into a new table `ORDERS_BACKFILL`, then return its row count. It should match `good_load_rows`.

In [ ]:
-- Your turn: zero-copy CLONE the pre-overwrite state into ORDERS_BACKFILL, then count its rows


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_BACKFILL
    CLONE {{my_db}}.TT_LAB.ORDERS_STAGING BEFORE(statement => $bad_overwrite);

    SELECT COUNT(*) AS backfill_rows FROM {{my_db}}.TT_LAB.ORDERS_BACKFILL;

A point-in-time `CLONE` is zero-copy, so materialising a past state as its own table is instant and costs no extra storage until it diverges.
</details>

> :mag_right: **Review the output.** You now have an independent `ORDERS_BACKFILL` table holding the historical state — created instantly, consuming no extra storage until it diverges from its source. That's the zero-copy clone earning its keep.

### Swap the recovered table in — atomically. 🔁

You've verified the clone. Now make it the live staging table. Snowflake's [`ALTER TABLE … SWAP WITH`](https://docs.snowflake.com/en/sql-reference/sql/alter-table) exchanges two tables — names, data, and metadata — in a **single atomic transaction**. There's never an instant where `ORDERS_STAGING` doesn't exist, so downstream models reading it mid-swap are never disrupted. After the swap the good load *is* `ORDERS_STAGING`; the thin bad slice lands in `ORDERS_STAGING_RECOVERED`, which you drop.

In [ ]:
-- Atomic swap: ORDERS_STAGING and the recovered clone trade places in ONE transaction,
-- so downstream jobs reading ORDERS_STAGING are never disrupted mid-swap.
ALTER TABLE {{my_db}}.TT_LAB.ORDERS_STAGING
  SWAP WITH {{my_db}}.TT_LAB.ORDERS_STAGING_RECOVERED;

-- After the swap, the bad slice now sits in ORDERS_STAGING_RECOVERED — drop that leftover.
DROP TABLE {{my_db}}.TT_LAB.ORDERS_STAGING_RECOVERED;

SELECT COUNT(*) AS final_rows FROM {{my_db}}.TT_LAB.ORDERS_STAGING;


> :mag_right: **Review the output.** `final_rows` matches `good_load_rows` — staging is rolled back to the last good load. Investigate → look back → recover → verify → swap. That's the pipeline rollback loop.

> :bulb: **Why `SWAP WITH` and not `DROP` + `RENAME`?** Dropping then renaming leaves a window where `ORDERS_STAGING` doesn't exist — and a downstream job hitting it in that instant fails. `ALTER TABLE … SWAP WITH …` swaps the two tables in a **single atomic transaction**, so the name always resolves. That matters most in a pipeline, where other jobs read the table continuously.

> 🧑‍💻 **Why this matters to you as a data engineer.** In a pipeline the swap is the critical move — `ALTER TABLE … SWAP WITH` is atomic, so models reading `ORDERS_STAGING` by name never hit a gap or a missing-table error mid-recovery. You can roll back a bad 2 AM load without scheduling downtime or coordinating with every downstream consumer.

### One more tool: `UNDROP`. ↩️

`CLONE … BEFORE` recovers *row-level* damage from a bad load. But if a cleanup job drops the **whole staging table**, there's a simpler rescue within the retention window: [`UNDROP`](https://docs.snowflake.com/en/sql-reference/sql/undrop-table).

First, drop the table and try to query it — you'll see the failure. Then, in the next cell, recover it with a single command.

> :point_right: **Step 1 — drop it.** Run the cell below. The `SELECT` will fail because the table no longer exists — that's the point. You'll fix it in the next cell.

In [ ]:
-- The accident: a cleanup job drops the whole staging table
DROP TABLE {{my_db}}.TT_LAB.ORDERS_STAGING;

-- Try to read it — this WILL fail
SELECT COUNT(*) FROM {{my_db}}.TT_LAB.ORDERS_STAGING;


> ⛔ **Expected error: table does not exist.** That's what a dropped table looks like to anyone querying it. Now recover it — fill in the blank below with the single command that restores a dropped table.

In [ ]:
___ TABLE {{my_db}}.TT_LAB.ORDERS_STAGING;   -- one-statement recovery of a dropped table

SELECT COUNT(*) AS rows_after_undrop FROM {{my_db}}.TT_LAB.ORDERS_STAGING;


<details>
<summary>📖 Reference solution</summary>

    UNDROP TABLE {{my_db}}.TT_LAB.ORDERS_STAGING;

`UNDROP` reverses a `DROP` within the retention window — a table (or schema, or database) comes back exactly as it was, in one statement. Use it for a dropped *object*; use `AT`/`BEFORE` + `CLONE` for row-level damage inside a table.
</details>

> :mag_right: **Review the output.** `rows_after_undrop` matches your last-good row count — the table came back intact, in one statement. You saw the failure (table gone), then the fix (`UNDROP`) — that's the full cycle. Use `UNDROP` for a dropped *object*; use `AT`/`BEFORE` + `CLONE` for row-level damage inside a table that still exists.

### See the whole story in your query history. 🧾

You just dropped and restored the staging table — and every statement you ran is recorded. `INFORMATION_SCHEMA.QUERY_HISTORY` lets you read your own recent activity back as a mini **audit trail**. Query it for anything that touched `ORDERS_STAGING` and you'll see the incident and its recovery in order: the bad load, the `CLONE`, the `DROP`, and the `UNDROP` — newest first.

In [ ]:
-- The audit trail: every recent statement that touched ORDERS_STAGING, newest first.
-- INFORMATION_SCHEMA.QUERY_HISTORY is a table function — wrap it in TABLE(...).
SELECT QUERY_ID, QUERY_TYPE, EXECUTION_STATUS, START_TIME, QUERY_TEXT
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 200))
WHERE QUERY_TEXT ILIKE '%ORDERS_STAGING%'
ORDER BY START_TIME DESC;


> :mag_right: **Review the output.** Read top-to-bottom (newest first) and you can retrace the whole incident. Your `DROP` shows as `QUERY_TYPE = DROP` — and here's the neat part: your `UNDROP` shows as **`RESTORE`**, Snowflake's internal name for the operation. The clone and load statements are all here too. Reconstructing *what happened* is the first move in any recovery, and `QUERY_HISTORY` is where you do it.

> :bulb: **`QUERY_HISTORY` is scoped to you.** `INFORMATION_SCHEMA.QUERY_HISTORY` returns *your* recent queries (last 7 days). Admins use the account-wide [`SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_history) view to see across all users — same idea, wider lens.

---
## Verify Your Work

The check below confirms your staging table has been rolled back to its last good load — the same number of rows as the shared source — and that the recovered clone was swapped in cleanly.

In [ ]:
all_passed = True
try:
    src = session.sql('SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS').collect()[0][0]
    final = session.sql(f'SELECT COUNT(*) FROM {my_db}.TT_LAB.ORDERS_STAGING').collect()[0][0]
    ok = final == src
    print(('✅' if ok else '⛔') + f' ORDERS_STAGING rolled back: {final} rows (source has {src})')
    if not ok:
        all_passed = False
        print('   Re-run Sections 4–5 — the recovered clone should match the source count.')
except Exception as e:
    print(f'⛔ Rollback check ERROR — {e}')
    all_passed = False

# The recovered clone should have been renamed IN (so the intermediate table is gone)
try:
    leftover = len(session.sql(f"SHOW TABLES LIKE 'ORDERS_STAGING_RECOVERED' IN SCHEMA {my_db}.TT_LAB").collect())
    ok = leftover == 0
    print(('✅' if ok else '⛔') + f' Recovered clone swapped in (no leftover table): {"clean" if ok else "still present"}')
    if not ok:
        all_passed = False
except Exception as e:
    print(f'⛔ Swap check ERROR — {e}')
    all_passed = False

if all_passed:
    print('\n🎉 All checks passed — you rolled back a bad overwrite with Time Travel.')


---
### Check your understanding. 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Time Travel retains history for a **retention period** (1 day default; up to 90 on Enterprise+) — roll back a bad load without re-running the pipeline
- **`INSERT OVERWRITE`** replaces rows atomically, but the prior version stays in Time Travel — recover it with **`BEFORE(statement => query_id)`**
- A point-in-time **`CLONE`** is zero-copy — recover the good state into a new table, **verify it**, then swap it in atomically with **`ALTER TABLE … SWAP WITH …`** so downstream jobs keep the same name and never see a missing table
- Clones aren't just for recovery — materialise any past state as a **backfill** table instantly, with no extra storage
- Dropped a whole table? Reach for **`UNDROP`** within the retention window
- **CoCo is your recovery partner** — it can find the offending job and draft the rollback SQL

**The Other Personas:**
- The **Data Analyst** notebook recovers a curated analysis table from a bad **`DELETE`**
- The **Platform Admin** notebook recovers from an accidental **`DROP SCHEMA`** and tunes retention policy

> :bulb: Each persona lab is independent and uses its own objects — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.